https://tutorials.pytorch.kr/beginner/basics/quickstart_tutorial.html
파이토치 빠른시작

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

# 데이터 작업

In [13]:
train_data = datasets.FashionMNIST(root='data', train=True, download=True, transform=ToTensor())
test_data = datasets.FashionMNIST(root='data', train=False, download=True, transform=ToTensor())

In [14]:
train_dataloader = DataLoader(train_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

for X, y in test_dataloader:
    print(f'shape of X[N,C,H,W]: {X.shape}')
    print(f'shape of y: {y.shape}: {y.dtype}')
    break

shape of X[N,C,H,W]: torch.Size([64, 1, 28, 28])
shape of y: torch.Size([64]): torch.int64


# 모델 만들기

In [15]:
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Using {device} device')

Using cuda device


In [16]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28,512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [17]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


# 모델 매개변수 최적화

In [18]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

In [19]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    for batch, (X,y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # 예측 오류 계산
        pred = model(X)
        loss = loss_fn(pred, y)

        # 역전파
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch+1) * len(X)
            print(f'loss: {loss:>.7f}, [{current:>5d}/{size:>5d}]')

In [20]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0,0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f'test error: Accuracy: {(100*correct):>0.1f}%, avg loss: {test_loss:>8f}')

In [21]:
epochs = 5
for t in range(epochs):
    print(f'epochs {t+1} ------------------')
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)

print('Done!!')

epochs 1 ------------------
loss: 2.3006799, [   64/60000]
loss: 2.2908826, [ 6464/60000]
loss: 2.2710314, [12864/60000]
loss: 2.2667232, [19264/60000]
loss: 2.2445097, [25664/60000]
loss: 2.2109065, [32064/60000]
loss: 2.2252102, [38464/60000]
loss: 2.1875124, [44864/60000]
loss: 2.1917706, [51264/60000]
loss: 2.1557939, [57664/60000]
test error: Accuracy: 42.8%, avg loss: 2.151313
epochs 2 ------------------
loss: 2.1655653, [   64/60000]
loss: 2.1550190, [ 6464/60000]
loss: 2.0957184, [12864/60000]
loss: 2.1155131, [19264/60000]
loss: 2.0549991, [25664/60000]
loss: 1.9924371, [32064/60000]
loss: 2.0284324, [38464/60000]
loss: 1.9407146, [44864/60000]
loss: 1.9572712, [51264/60000]
loss: 1.8891320, [57664/60000]
test error: Accuracy: 50.3%, avg loss: 1.881367
epochs 3 ------------------
loss: 1.9181299, [   64/60000]
loss: 1.8856651, [ 6464/60000]
loss: 1.7650757, [12864/60000]
loss: 1.8134524, [19264/60000]
loss: 1.6922013, [25664/60000]
loss: 1.6479460, [32064/60000]
loss: 1.676646

# 모델 저장

In [22]:
torch.save(model.state_dict(), '20250806model.pth')

# 모델 로드

In [23]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load('20250806model.pth'))

<All keys matched successfully>

In [32]:
train_data.classes

['T-shirt/top',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']